# PROC SUMMARY로 통신사 수익보증 요약 큐브 구축하기

## 요약

무선통신사의 수익보증(revenue-assurance) 팀은 한 달치 가입자-일별 청구 기록을 압축된 요약 큐브로 사전 집계하여, 분석가가 원본 팩트 테이블을 다시 스캔하지 않고도 지역, 요금제 등급, 통화유형별로 드릴다운할 수 있게 한다. `PROC SUMMARY`는 100건의 가입자-일 레코드를 다차원 셀 집합으로 롤업한다: 가장 세밀한 그레인(지역 x 요금제등급 x 통화유형)은 가능한 27개 셀 중 25개를 채우며, 명명된 서브큐브는 분석가가 가장 많이 조회하는 한계값(marginal)을 제공한다. 이번 표본 월에는 통신사가 3개 활성 지역에서 총 \$222.78를 정산했으며, 남부(\$97.44)와 동부(\$86.94)가 합쳐서 매출의 83%를 차지하고 북부는 \$38.40으로 뒤처진다. 가장 풍부한 단일 서브큐브는 Plus 등급 음성 서비스(18건에 걸쳐 \$59.06)이며, 셀을 레코드당 수익률로 순위 매기면 데이터 사용 셀이 누수 감사(leakage audit)를 위한 가장 가치 있는 타깃으로 부상한다(최고 수익률 레코드당 \$7.87). 아래의 모든 수치는 실행된 출력에서 직접 읽은 것이다.

## 데이터 출처

| 데이터셋 | 행 수 | 그레인 | 주요 변수 |
|---------|------|-------|---------------|
| `work.cdr_billing` | 100 | 가입자-일 사용 요약 1행 | `region`(동부/남부/북부), `plan_tier`(선불/베이직/플러스), `call_type`(음성/문자/데이터), `device_class`, `bill_month`, `revenue`, `call_minutes`, `data_mb`, `subscriber_wt` |
| `work.cube_nway` | 25 | 비어있지 않은 (지역 x plan_tier x call_type) 셀 1행 | `_FREQ_`, `rev_total`, `rev_mean`, `rev_max`, `min_total`, `data_total` |
| `work.cube_hier` | 22 | 명명된 드릴 서브큐브의 셀 1행 | `_TYPE_`, `_FREQ_`, `rev_total` |

모든 데이터는 `call streaminit()` / `rand()`로 인라인 생성되며, 외부 파일이나 네트워크 접근이 필요하지 않다. 이 환경은 라이선스가 없는 상태로 실행되므로 기록되는 데이터셋은 100건으로 제한된다 — 이 시나리오는 그 상한선 이내에서 큐브가 완전히 채워지도록 설계되었다.

## 원시 통화상세기록(CDR)에서 드릴 가능한 큐브까지

무선통신사는 매일 수백만 건의 사용 이벤트에 걸쳐 매출을 정산한다. 수익보증 분석가가 *"지난달 남부 지역의 Plus 등급 음성 매출은 얼마였는가?"*와 같은 질문에 매번 원본 팩트 테이블을 재스캔하지 않고도 답할 수 있도록, 데이터를 압축된 요약 큐브로 **사전 집계**한다.

`PROC SUMMARY`는 바로 이 작업을 위한 Base SAS의 주력 도구다: 하나 이상의 `CLASS` 차원으로 평면 팩트 테이블을 그룹화하고, 요청된 통계량을 출력 데이터셋에 기록하며, 각 행에 `_TYPE_`(활성화된 차원)과 `_FREQ_`(셀 뒤의 레코드 수)를 태그한다. 그 출력 데이터셋이 바로 다차원 큐브다 — OLAP 도구가 노출하는 것과 동일한 롤업 구조가 일반 SAS 데이터셋으로 구체화되어, 인쇄하고 조인하고 슬라이스할 수 있다.

이 노트북은:

1. 현실적인 한 달치 가입자-일 청구 기록을 생성한다.
2. `PROC SUMMARY NWAY`로 가장 세밀한 그레인의 큐브(지역 x 요금제등급 x 통화유형)를 구축한다.
3. `TYPES` 문으로 명명된 **드릴 서브큐브**를 구체화한다.
4. `WEIGHT`로 매출을 가입자 기반에 투영하고, 한 지역으로 드릴다운하며, 누수 감사(triage)를 위해 셀을 레코드당 수익률로 순위 매긴다.

## 1단계 - 합성 통화상세기록 / 청구 데이터 생성

각 행은 하루 동안 한 가입자가 한 서비스를 사용한 내역을 요약한다. 재현성을 위해 `call streaminit`을 사용하고, `rand()`로 그럴듯한 분포를 뽑는다: 매출과 사용량은 요금제 등급에 비례해 확대되며, 음성 매출은 과금 분(分)을 따라가고 데이터 매출은 메가바이트를 따라간다. 각 `RAND('table', ...)`에는 카테고리당 하나의 확률이 주어져, 100건 표본에서 모든 지역, 등급, 통화유형이 등장하도록 한다. 나중에 가중 지표를 시연할 수 있도록 소규모 `subscriber_wt` 설문 가중치도 부여한다.

In [1]:
데이터 work.cdr_billing;
    호출 streaminit(20260131);
    길이 region $12 plan_tier $12 call_type $12 device_class $12 bill_month $7;
    라벨 revenue       = "정산 매출 (USD)"
          call_minutes  = "과금 음성 통화 분"
          data_mb       = "데이터 사용량 (MB)"
          subscriber_wt = "가입자 설문 가중치";

    반복 i = 1 까지 100;
        /* --- 차원: 합이 1.0이 되는 카테고리별 확률 --- */
        r = rand("table", 0.40, 0.33, 0.27);
        만약 r = 1 이면 region = "동부";
        아니면 만약 r = 2 이면 region = "남부";
        아니면 region = "북부";

        p = rand("table", 0.30, 0.40, 0.30);
        만약 p = 1 이면 plan_tier = "선불";
        아니면 만약 p = 2 이면 plan_tier = "베이직";
        아니면 plan_tier = "플러스";

        c = rand("table", 0.50, 0.30, 0.20);
        만약 c = 1 이면 call_type = "음성";
        아니면 만약 c = 2 이면 call_type = "문자";
        아니면 call_type = "데이터";

        만약 rand("uniform") < 0.55 이면 device_class = "스마트";
        아니면 device_class = "피처";

        bill_month = "2026-01";

        /* --- 측정값: 등급과 서비스에 따라 스케일 --- */
        tier_mult = (plan_tier = "선불")*0.7
                  + (plan_tier = "베이직")*1.0
                  + (plan_tier = "플러스")*1.7;

        call_minutes = round((call_type = "음성")
                       * rand("gamma", 2.0) * 18 * tier_mult, 0.1);
        data_mb      = round((call_type = "데이터")
                       * rand("gamma", 1.5) * 220 * tier_mult, 0.1);

        base_rev = 0.05*call_minutes + 0.010*data_mb
                 + (call_type = "문자") * rand("poisson", 30) * 0.03;
        revenue = round(base_rev * (0.85 + 0.30*rand("uniform")), 0.01);

        subscriber_wt = round(0.8 + rand("uniform")*1.6, 0.01);

        출력;
    종료;
    제거 i r p c tier_mult base_rev;
실행;

proc print data=work.cdr_billing(obs=8) label noobs;
    title "표본 가입자-일 청구 기록";
run;

                                                     표본 가입자-일 청구 기록                                                     

region  plan_tier  call_type  device_class  bill_month                과금 음성 통화 분              데이터 사용량 (MB)          정산 매출 (USD)                  가입자 설문 가중치
북부      플러스        문자         스마트           2026-01                            0                         0                 0.67                        1.13
남부      선불         문자         피처            2026-01                            0                         0                 0.94                        1.42
남부      플러스        문자         스마트           2026-01                            0                         0                 0.99                        0.86
남부      플러스        문자         스마트           2026-01                            0                         0                 1.01                        1.03
남부      플러스        음성         스마트           2026-01                        103.4                  


NOTE: DATA work.cdr_billing


NOTE: Wrote work.cdr_billing (100 rows, 9 columns).
NOTE: DATA elapsed:
  wall  0.03 seconds
  cpu   0.03 seconds
NOTE: PROC PRINT data=work.cdr_billing

NOTE: PROC PRINT completed: 8 observations printed, 9 variables


## 2단계 - PROC SUMMARY NWAY로 가장 세밀한 그레인의 큐브 구축

`NWAY`는 모든 `CLASS` 차원 조합 중 가장 세밀한 단일 조합만 남긴다 - 여기서는 채워진 모든 (지역 x plan_tier x call_type) 셀이다. 각 셀에 대해 매출의 `SUM`, `MEAN`, `MAX`와 총 통화 분 및 메가바이트를 저장하여, 분석가가 셀당 총 매출을 읽고 레코드당 평균을 도출하며 가장 큰 단일 청구를 찾아낼 수 있게 한다. `_FREQ_`는 각 셀 뒤에 몇 건의 가입자-일이 있는지 기록한다. NWAY 그레인에서는 모든 행이 같은 타입을 가지므로 여기서는 `_TYPE_`을 제거한다.

In [2]:
proc summary data=work.cdr_billing nway;
    class region plan_tier call_type;
    var revenue call_minutes data_mb;
    output out=work.cube_nway(drop=_type_)
           sum(revenue)=rev_total  mean(revenue)=rev_mean  max(revenue)=rev_max
           sum(call_minutes)=min_total
           sum(data_mb)=data_total;
run;

proc print data=work.cube_nway(obs=12) noobs;
    title "NWAY 큐브 셀 (지역 * plan_tier * call_type)";
    format rev_total rev_mean rev_max min_total data_total comma10.2;
run;

                                         NWAY 큐브 셀 (지역 * plan_tier * call_type)                                         

REGION  PLAN_TIER  CALL_TYPE  _FREQ_  REV_TOTAL  REV_MEAN  REV_MAX  MIN_TOTAL  DATA_TOTAL
남부      베이직        데이터             3      12.02      4.01     5.98       0.00    1,368.50
남부      베이직        문자              2       2.01      1.00     1.14       0.00        0.00
남부      베이직        음성              9      22.51      2.50     4.76     451.80        0.00
남부      선불         데이터             5      12.34      2.47     4.79       0.00    1,170.40
남부      선불         문자              2       1.82      0.91     0.94       0.00        0.00
남부      선불         음성              3       2.59      0.86     1.89      49.90        0.00
남부      플러스        데이터             2      11.92      5.96    10.16       0.00    1,122.90
남부      플러스        문자              5       5.16      1.03     1.35       0.00        0.00
남부      플러스        음성              8      27.07      3.38     5.99  


NOTE: PROC MEANS
NOTE: Output dataset work.cube_nway has 25 observations and 9 variables.
NOTE: PROC MEANS statement used.
NOTE: PROC PRINT data=work.cube_nway

NOTE: PROC PRINT completed: 12 observations printed, 9 variables


## 3단계 - TYPES로 명명된 드릴 서브큐브 구체화

OLAP 큐브는 분석가가 가장 많이 탐색하는 롤업을 미리 저장해 둔다. `TYPES` 문이 바로 그 역할을 한다: 각 항이 `PROC SUMMARY`에 서브큐브 하나를 생성하도록 요청한다. 총합계 `()`, `region` 한계값, 그리고 이원(two-way) `region*plan_tier`와 `call_type*plan_tier` 서브큐브를 요청한다 - 매출 대시보드가 노출할 드릴 경로들이다. SAS는 각 서브큐브에 `_TYPE_` 코드(`CLASS` 목록에 대한 비트마스크)를 태그하여, 단일 출력 데이터셋이 계층구조의 모든 수준을 담도록 한다.

In [3]:
proc summary data=work.cdr_billing;
    class region plan_tier call_type;
    var revenue;
    types () region region*plan_tier call_type*plan_tier;
    output out=work.cube_hier
           sum(revenue)=rev_total;
run;

proc print data=work.cube_hier noobs;
    title "서브큐브 롤업: 총합계, 지역, 지역*등급, 통화유형*등급";
    var _type_ region plan_tier call_type _freq_ rev_total;
    format rev_total comma10.2;
run;

                                            서브큐브 롤업: 총합계, 지역, 지역*등급, 통화유형*등급                                            

_TYPE_  REGION  PLAN_TIER  CALL_TYPE  _FREQ_  REV_TOTAL
     0                                   100     222.78
     3          베이직        데이터             8      38.06
     3          베이직        문자              8       8.03
     3          베이직        음성             20      42.33
     3          선불         데이터             7      14.50
     3          선불         문자              7       6.82
     3          선불         음성             16      24.77
     3          플러스        데이터             3      17.46
     3          플러스        문자             13      11.75
     3          플러스        음성             18      59.06
     4  남부                                39      97.44
     4  동부                                38      86.94
     4  북부                                23      38.40
     6  남부      베이직                       14      36.54
     6  남부      선불                    


NOTE: PROC MEANS
NOTE: Output dataset work.cube_hier has 22 observations and 6 variables.
NOTE: PROC MEANS statement used.
NOTE: PROC PRINT data=work.cube_hier

NOTE: PROC PRINT completed: 22 observations printed, 6 variables


## 4단계 - 가중 투영, 지역 드릴다운, 누수 감사

수익보증 팀이 실제로 큐브에 대해 수행하는 세 가지 조회:

- **가중 투영.** `region*plan_tier` 요약에 `WEIGHT=subscriber_wt`를 부여하면, 원시 표본 행이 아니라 표본이 대표하는 전체 가입자 기반으로 확대된 매출을 보고한다.
- **드릴다운.** NWAY 큐브를 한 지역으로 필터링하면 계층구조의 한 가지가 등급별-서비스별 세부내역으로 확장된다 - 여기서는 남부.
- **누수 감사(triage).** 셀을 레코드당 평균 매출로 정렬하면 최고 수익률 셀이 드러난다. 빈도가 낮으면서 수익률이 높은 셀이야말로 감사가 오요율 산정 또는 매출 누수를 정밀 조사하는 대상이다.

In [4]:
/* 가입자 기반으로 투영된 가중 매출 */
proc summary data=work.cdr_billing nway;
    class region plan_tier;
    var revenue;
    weight subscriber_wt;
    output out=work.cube_wt(drop=_type_ _freq_)
           sum(revenue)=rev_weighted  n(revenue)=records;
run;

proc print data=work.cube_wt noobs;
    title "지역 * 요금제등급별 가중 매출 (가입자 기반 투영)";
    format rev_weighted comma10.2;
run;

/* 남부 지역 가지로 드릴다운 */
proc print data=work.cube_nway noobs;
    where region = "남부";
    title "남부 드릴다운: 등급 및 통화유형별 매출 셀";
    var plan_tier call_type _freq_ rev_total rev_mean;
    format rev_total rev_mean comma10.2;
run;

/* 누수 감사를 위해 셀을 레코드당 수익률로 순위 매기기 */
proc sort data=work.cube_nway out=work.cube_ranked;
    by descending rev_mean;
run;

proc print data=work.cube_ranked(obs=6) noobs;
    title "레코드당 수익(수익률)이 가장 높은 셀";
    var region plan_tier call_type _freq_ rev_mean rev_max;
    format rev_mean rev_max comma10.2;
run;

                                             지역 * 요금제등급별 가중 매출 (가입자 기반 투영)                                              

REGION  PLAN_TIER  REV_WEIGHTED  RECORDS
남부      베이직               58.63       14
남부      선불                27.77       10
남부      플러스               56.29       15
동부      베이직               50.85       15
동부      선불                29.77       11
동부      플러스               59.59       12
북부      베이직               18.30        7
북부      선불                20.56        9
북부      플러스               22.42        7

                                                남부 드릴다운: 등급 및 통화유형별 매출 셀                                                

PLAN_TIER  CALL_TYPE  _FREQ_  REV_TOTAL  REV_MEAN
베이직        데이터             3      12.02      4.01
베이직        문자              2       2.01      1.00
베이직        음성              9      22.51      2.50
선불         데이터             5      12.34      2.47
선불         문자              2       1.82      0.91
선불         음성              3       2.59      


NOTE: PROC MEANS
NOTE: Output dataset work.cube_wt has 9 observations and 4 variables.
NOTE: PROC MEANS statement used.
NOTE: PROC PRINT data=work.cube_wt

NOTE: PROC PRINT completed: 9 observations printed, 4 variables
NOTE: PROC PRINT data=work.cube_nway

NOTE: PROC PRINT completed: 9 observations printed, 5 variables
NOTE: PROC SORT data=work.cube_nway

NOTE: Unlicensed mode - input limited to 100 observations.
NOTE: Read 25 rows from work.cube_nway.
NOTE: Wrote work.cube_ranked (25 rows, 9 columns).
NOTE: PROC SORT statement used.
NOTE: PROC PRINT data=work.cube_ranked

NOTE: PROC PRINT completed: 6 observations printed, 6 variables


## 결과 해석

요약 큐브는 100건의 원시 가입자-일 행을 압축된 사전 집계 셀 집합으로 바꾸어, 팩트 테이블을 다시 스캔하지 않고도 드릴다운 질문에 즉시 답할 수 있게 한다:

- **매출이 어디에 있는가.** `region` 한계값(`_TYPE_=4`)은 남부가 \$97.44, 동부가 \$86.94를 정산했음을 보여준다 - 이번 달 \$222.78 중 합쳐서 83%다 - 반면 북부는 \$38.40을 기여했다. `call_type*plan_tier` 서브큐브(`_TYPE_=3`) 내에서는 Plus 등급 음성이 18건에 걸쳐 \$59.06로 가장 풍부한 단일 셀이며, Basic 등급 음성이 \$42.33으로 그 다음이다.
- **가중 투영.** 설문 가중치를 적용하면 가입자 비중이 더 큰 요금제 쪽으로 순위가 재편된다: 동부-플러스(\$59.59)와 남부-베이직(\$58.63)이 투영된 `region*plan_tier` 매출을 주도하며, 이는 가중되지 않은 지역 총계와는 다른 그림이자, 노출을 산정할 때 표본 매출이 아니라 투영된 매출을 보고해야 한다는 상기시킴이다.
- **레코드당 수익률과 누수 감사.** NWAY 셀을 `rev_mean`으로 순위 매기면 데이터 사용 셀이 상위에 오른다 - 북부-베이직-데이터(레코드당 \$7.87)와 남부-플러스-데이터(레코드당 \$5.96) - 이는 데이터 중심 사용이 레코드당 가장 높은 매출을 만들어낸다는 것을 확인해준다. 빈도가 낮은(한두 건) 최상위 셀이야말로 수익보증 분석가가 고액 청구가 오류가 아니라 올바르게 요율 산정되었는지 확인하기 위해 원본 통화상세기록을 조회할 대상이다.

수익보증 팀에게 이 큐브는 편차 대시보드의 기반이 된다: (지역, 등급, 통화유형) 셀별로 정산된 매출을 요율표(rate-card) 예상 매출과 비교하면, 평균값 또는 가중 총계가 가장 크게 벗어나는 셀이 감사할 가치가 있는 누수 사례가 된다. 전체 구조가 일반 SAS 데이터셋이므로, 다음 달의 큐브도 같은 Base SAS 도구로 합집합(union), 차이 비교, 또는 요율표와의 조인이 가능하다.